# **Topic Modelling**

### USING LDA

In [ ]:
# 1. (Opsional) Install dulu library yang diperlukan:
#    pip install pandas scikit-learn langdetect

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

# Asumsi: File CSV bernama 'preprocessed_data.csv' berada di direktori yang sama dengan skrip Python
df = pd.read_csv(r"D:\Internship\AIBeecara\PROJECT TEST\data_stemm_2.csv")

# Menampilkan beberapa baris pertama dari DataFrame untuk verifikasi
print(df.head())

# 3. Deteksi bahasa (menghasilkan 'en', 'id', atau 'unknown')
def detect_lang(text):
    try:
        return detect(text)
    except:
        return 'unknown'

df['lang'] = df['stemmed'].apply(detect_lang)

# 4. Fungsi untuk menjalankan LDA pada subset bahasa tertentu
def run_lda(texts, n_topics=5, max_df=0.9, min_df=5, n_top_words=10):
    """
    texts      : list of str (dokumen yang akan di-LDA)
    n_topics   : jumlah topik
    max_df     : abaikan kata yang muncul di > max_df proporsi dokumen
    min_df     : abaikan kata yang muncul di < min_df dokumen
    """
    # Buat Document-Term Matrix
    vect = CountVectorizer(max_df=max_df, min_df=min_df)
    dtm = vect.fit_transform(texts)
    
    # Fit LDA
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method='batch'
    )
    lda.fit(dtm)
    
    # Tampilkan top words per topic
    feature_names = vect.get_feature_names_out()
    for idx, topic in enumerate(lda.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic #{idx+1}: {', '.join(top_words)}")

# 5. Jalankan untuk masing-masing bahasa
for lang_code in ['en', 'id']:
    subset = df[df['lang'] == lang_code]['stemmed'].tolist()
    if subset:
        print(f"\n=== Topics for language: {lang_code} ===")
        run_lda(
            texts=subset,
            n_topics=5,       # misalnya 5 topik per bahasa
            max_df=0.8,       # abaikan kata yang terlalu umum
            min_df=5,         # abaikan kata yang terlalu jarang
            n_top_words=10    # 10 kata teratas tiap topik
        )
    else:
        print(f"\n(no documents for language '{lang_code}')")


### USING LDA+BERT

In [23]:
# 1. (jika belum) install dependencies:
#    pip install pandas scikit-learn langdetect textblob transformers torch

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from transformers import pipeline

# 2. Load data
df = pd.read_csv('data_stemm.csv')

# 3. Deteksi bahasa aman
def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip():
        return 'unknown'
    try:
        return detect(text)
    except:
        return 'unknown'

df['lang'] = df['stemmed'].apply(detect_lang_safe)

# 4. Filter komentar valid (hanya 'en' & 'id', non-empty)
df = df[df['lang'].isin(['en','id'])].copy()
df = df[df['stemmed'].str.strip() != '']

# 5. Vectorizer + LDA (5 topik)
vect = CountVectorizer(max_df=0.8, min_df=5)
dtm  = vect.fit_transform(df['stemmed'])
lda  = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(dtm)

# 6. Assign topik dominan
topic_dist = lda.transform(dtm)           # (n_docs, n_topics)
df['dominant_topic'] = topic_dist.argmax(axis=1)

# 7. Siapkan sentiment analyzers
#    – English pakai TextBlob
def sentiment_en(text):
    p = TextBlob(text).sentiment.polarity
    return 'positive' if p>0.1 else 'negative' if p<-0.1 else 'neutral'

#    – Indonesian pakai Roberta-light model
sent_id_pipe = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)
def sentiment_id(text):
    # pipe mengembalikan [{'label': 'POSITIVE'|'NEGATIVE'|'NEUTRAL', 'score':...}]
    out = sent_id_pipe(text[:512])[0]
    lbl = out['label'].lower()
    # sesuaikan:
    if 'neg' in lbl: return 'negative'
    if 'pos' in lbl: return 'positive'
    return 'neutral'

# 8. Terapkan sentiment sesuai bahasa
def get_sent(row):
    return sentiment_en(row['stemmed']) if row['lang']=='en' else sentiment_id(row['stemmed'])

df['sentiment'] = df.apply(get_sent, axis=1)

# 9. Ringkasan: hitung distribusi per (topik, sentiment) + label mayoritas
summary = (
    df
    .groupby(['dominant_topic','sentiment'])
    .size()
    .unstack(fill_value=0)
)
summary['topic_sentiment'] = summary.idxmax(axis=1)

print(summary)


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

sentiment       negative  neutral  positive topic_sentiment
dominant_topic                                             
0                     40      118       136        positive
1                     39       10        18        negative
2                    150       90       662        positive
3                     35      105       284        positive
4                     21       89       170        positive


In [1]:
# 1. Instalasi (jika perlu)
# pip install pandas scikit-learn langdetect textblob transformers torch

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from transformers import pipeline

# 2. Load & deteksi bahasa (tangani NaN)
df = pd.read_csv('data_stemm.csv')
def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip(): return 'unknown'
    try: return detect(text)
    except: return 'unknown'
df['lang'] = df['stemmed'].apply(detect_lang_safe)
df = df[df['lang'].isin(['en','id'])].copy()
df = df[df['stemmed'].str.strip() != '']

# 3. Siapkan sentiment analyzers
# English via TextBlob
def sentiment_en(txt):
    p = TextBlob(txt).sentiment.polarity
    return 'positive' if p>0.1 else 'negative' if p<-0.1 else 'neutral'

# Indonesian via Roberta classifier
sent_id = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)
def sentiment_id(txt):
    out = sent_id(txt[:512])[0]
    lbl = out['label'].lower()
    if 'neg' in lbl: return 'negative'
    if 'pos' in lbl: return 'positive'
    return 'neutral'

# 4. Fungsi helper: process per bahasa
def analyze_language(df, lang_code, n_topics=5, n_top_words=10):
    sub = df[df['lang']==lang_code].copy()
    texts = sub['stemmed'].tolist()
    
    # a) Vectorize & fit LDA
    vect = CountVectorizer(max_df=0.8, min_df=5)
    dtm  = vect.fit_transform(texts)
    lda  = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(dtm)
    
    # b) Assign topik dominan
    topic_dist = lda.transform(dtm)
    sub['dominant_topic'] = topic_dist.argmax(axis=1)
    
    # c) Hitung sentiment per dokumen
    if lang_code=='en':
        sub['sentiment'] = sub['stemmed'].apply(sentiment_en)
    else:
        sub['sentiment'] = sub['stemmed'].apply(sentiment_id)
    
    # d) Ringkasan sentiment per topik
    summary = (sub
               .groupby(['dominant_topic','sentiment'])
               .size()
               .unstack(fill_value=0))
    summary['topic_sentiment'] = summary.idxmax(axis=1)
    
    # e) Cetak top words + topic_sentiment
    feat = vect.get_feature_names_out()
    print(f"\n=== Language: {lang_code} ===")
    for t in range(n_topics):
        # top words
        top_idx = lda.components_[t].argsort()[:-n_top_words-1:-1]
        topw    = [feat[i] for i in top_idx]
        sent_lbl= summary.loc[t,'topic_sentiment']
        print(f"Topic {t+1} ({sent_lbl}): {', '.join(topw)}")
    
    # f) Tampilkan tabel summary jika mau
    print("\nDistribution per topic & sentiment:")
    print(summary)
    return sub, summary

# 5. Jalankan untuk 'en' dan 'id'
sub_en, summary_en = analyze_language(df, 'en', n_topics=5, n_top_words=10)
sub_id, summary_id = analyze_language(df, 'id', n_topics=5, n_top_words=10)



=== Language: en ===
Topic 1 (positive): app, ad, heart, learn, use, practic, get, lesson, free, chang
Topic 2 (positive): learn, languag, app, duolingo, lesson, use, new, help, like, great
Topic 3 (positive): lesson, ad, app, get, use, pay, everi, time, duolingo, xp
Topic 4 (positive): learn, word, app, languag, use, also, like, im, english, would
Topic 5 (positive): learn, word, app, like, lesson, time, languag, speak, duolingo, make

Distribution per topic & sentiment:
sentiment       negative  neutral  positive topic_sentiment
dominant_topic                                             
0                     20       75       141        positive
1                     12       61       176        positive
2                     28       81       107        positive
3                     15       35        49        positive
4                     20       64       118        positive

=== Language: id ===
Topic 1 (positive): ajar, banget, duolingo, nya, hati, kalo, bagus, ga, udah, aj